# Klasifikasi Lesi Kulit: ResNet-50 + Triplet Attention (Split HAM10k + ISIC 2019, Masker U-Net)

* Dataset: split `ham10k-and-isic-2019` (train/val/test bebas kebocoran, tidak di-split ulang).
* Preprocessing: pad-resize 224, normalisasi kontras, segmentasi soft-blend dengan masker.
* Masker: HAM10000 dari Tschandl, ISIC 2019 dari pseudo-mask U-Net (Val Dice 0,9368).
* Arsitektur, loss, dan fase pelatihan sama dengan `sk3-bigdat-data2017-3kelas-ak85`.
* Dioptimalkan untuk GPU A100: mixed precision, batch 64, data di RAM, augmentasi di GPU.
* Ubah `TASK` di sel konfigurasi: `irisan` (3 kelas) atau `binary` (2 kelas).

# Konfigurasi

In [ ]:
!pip install -q kagglehub
import os
from pathlib import Path

TASK = "irisan"   # "irisan" (3 kelas) atau "binary" (2 kelas)

IS_COLAB = os.path.exists("/content") and not os.path.exists("/kaggle/working")
WORK_DIR = Path("/content") if IS_COLAB else Path("/kaggle/working")


def get_dataset(slug):
    if IS_COLAB:
        import kagglehub
        return Path(kagglehub.dataset_download(f"nadiraanindita/{slug}"))
    return Path("/kaggle/input/datasets/nadiraanindita") / slug


DIR_SPLIT = get_dataset("ham10k-and-isic-2019")
DIR_HAM = get_dataset("dataset-riset-ham10k")
DIR_2019 = get_dataset("dataset-riset-isic-2019")
DIR_HAM_MASK = get_dataset("ham-segmentations-lesion-riset-saya")
DIR_PSEUDO = get_dataset("isic2019-pseudo-masks-zip")

NUM_WORKERS = 2      # jumlah worker Spark (local[N])
NUM_PARTITIONS = 2   # jumlah partisi data Spark
IMG_SIZE = 224
BATCH_SIZE = 64
BG_KEEP = 0.2        # sisa intensitas latar di luar lesi (0 = hitam penuh, 1 = tanpa segmentasi)
PROC_DIR = WORK_DIR / "processed_images"
PROC_DIR.mkdir(parents=True, exist_ok=True)
CKPT_PATH = str(WORK_DIR / "best_model_ak85.keras")

if TASK == "irisan":
    LABEL_COL, SUFFIX = "harmonized_label", "_3class.csv"
    LABEL_MAP = {"nv": "nevus", "mel": "melanoma", "bkl": "seborrheic_keratosis"}
else:
    LABEL_COL, SUFFIX = "binary_class", ".csv"
    LABEL_MAP = {"benign": "benign", "malignant": "malignant"}
NUM_CLASSES = len(LABEL_MAP)
print(f"Colab={IS_COLAB} | TASK={TASK} | kelas={list(LABEL_MAP.values())}")

# Inisialisasi spark

In [ ]:
!pip install pyspark -q

import os
import tensorflow as tf
from pyspark.sql import SparkSession

print("Daftar Perangkat Fisik GPU:")
print(tf.config.list_physical_devices('GPU'))
print("-" * 40)

tf.keras.mixed_precision.set_global_policy("mixed_float16")
print("Mixed precision aktif:", tf.keras.mixed_precision.global_policy().name)

spark = SparkSession.builder \
    .appName("SkinCancer_BigData_UNetMask") \
    .master(f"local[{NUM_WORKERS}]") \
    .config("spark.default.parallelism", str(NUM_PARTITIONS)) \
    .config("spark.driver.memory", "8g") \
    .config("spark.driver.maxResultSize", "2g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

print(f"Spark Session siap: {NUM_WORKERS} worker, {NUM_PARTITIONS} partisi.")

# Load dataset

In [ ]:
import os, re, time, zipfile, warnings
import pandas as pd
from pyspark import TaskContext

warnings.filterwarnings("ignore")
spark.sparkContext.setLogLevel("ERROR")


def safe_read_csv(path):
    for enc in ["utf-8", "latin-1", "cp1252"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue
    return pd.read_csv(path, encoding_errors="replace")


def load_split(name):
    hits = [p for p in DIR_SPLIT.rglob(f"{name}{SUFFIX}")]
    assert hits, f"{name}{SUFFIX} tidak ditemukan di {DIR_SPLIT}"
    df = safe_read_csv(hits[0])
    if "image_id" not in df.columns:
        df = df.rename(columns={next(c for c in df.columns if c.lower() in ("image", "isic_id")): "image_id"})
    if LABEL_COL not in df.columns and TASK == "binary" and "target_binary" in df.columns:
        df[LABEL_COL] = df["target_binary"].map({0: "benign", 1: "malignant"})
    assert LABEL_COL in df.columns, f"Kolom {LABEL_COL} tidak ada. Kolom tersedia: {list(df.columns)}"
    df["label"] = df[LABEL_COL].astype(str).str.strip().str.lower().map(LABEL_MAP)
    df["split"] = name
    if "source" not in df.columns:
        df["source"] = "unknown"
    return df[["image_id", "label", "source", "split"]].dropna(subset=["label"]).drop_duplicates("image_id")


df_split = pd.concat([load_split(n) for n in ["train", "val", "test"]], ignore_index=True)
print(f"[INFO] Manifest split: {len(df_split)} baris")
print(df_split.groupby(["split", "label"]).size().unstack(fill_value=0))


def index_files(root, exts):
    found = {}
    for r, _, files in os.walk(root):
        for f in files:
            stem, ext = os.path.splitext(f)
            if ext.lower() in exts:
                found.setdefault(re.sub(r"_segmentation$", "", stem), os.path.join(r, f))
    return found


image_index = {**index_files(DIR_2019, {".jpg", ".jpeg"}), **index_files(DIR_HAM, {".jpg", ".jpeg"})}

pseudo_zips = list(DIR_PSEUDO.rglob("*.zip"))
if pseudo_zips:
    pseudo_dir = WORK_DIR / "pseudo_masks"
    pseudo_dir.mkdir(exist_ok=True)
    for z in pseudo_zips:
        with zipfile.ZipFile(z) as zf:
            zf.extractall(pseudo_dir)
    pseudo_root = pseudo_dir
else:
    pseudo_root = DIR_PSEUDO
mask_pseudo = index_files(pseudo_root, {".png"})
mask_ham = index_files(DIR_HAM_MASK, {".png"})
mask_index = {**mask_pseudo, **mask_ham}
print(f"[INFO] Citra: {len(image_index)} | Masker HAM: {len(mask_ham)} | Pseudo-mask 2019: {len(mask_pseudo)}")

df_split["filepath"] = df_split["image_id"].map(image_index)
df_split["mask_path"] = df_split["image_id"].map(mask_index)
ok = df_split["filepath"].notna() & df_split["mask_path"].notna()
print("\n=== Baris dibuang (citra atau masker tidak ada) ===")
print(df_split[~ok].groupby(["source", "split"]).size().to_string() or "tidak ada")
df_final = df_split[ok].reset_index(drop=True)
print(f"[INFO] Siap diproses: {len(df_final)} dari {len(df_split)} baris")


def track_executor_time(iterator):
    start = time.time()
    n = sum(1 for _ in iterator)
    ctx = TaskContext.get()
    yield (ctx.partitionId() if ctx else -1, n, time.time() - start)


spark_df = spark.createDataFrame(df_final[["image_id", "label", "split"]])
print("\n=== BEBAN KERJA EXECUTOR SPARK ===")
for pid, n, t in spark_df.rdd.mapPartitions(track_executor_time).collect():
    print(f"Partisi {pid}: {n} baris, {t:.4f} detik")

In [ ]:
import matplotlib.pyplot as plt

print(f"=== RINGKASAN DATASET ({TASK.upper()}) ===")
print(f"Total citra: {len(df_final)}\n")
print(pd.crosstab(df_final["split"], df_final["label"], margins=True))
print()
print(pd.crosstab(df_final["source"], df_final["split"], margins=True))

counts = df_final["label"].value_counts()
plt.figure(figsize=(8, 4))
bars = plt.bar(counts.index, counts.values, color=["#2b5c8f", "#d95f02", "#7570b3"][:len(counts)])
plt.title(f"Distribusi Kelas ({TASK})")
plt.ylabel("Jumlah Citra")
plt.grid(axis="y", linestyle="--", alpha=0.6)
for b in bars:
    plt.text(b.get_x() + b.get_width() / 2, b.get_height(), f"{int(b.get_height()):,}", ha="center", va="bottom")
plt.tight_layout()
plt.show()

# preprosesing citra (resize, normalisasi kontras, segmentasi masker U-Net)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import time
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from pyspark.sql import SparkSession


def pad_and_resize(img, size, interp, border):
    h, w = img.shape[:2]
    m = max(h, w)
    top, left = (m - h) // 2, (m - w) // 2
    padded = cv2.copyMakeBorder(img, top, m - h - top, left, m - w - left, border)
    return cv2.resize(padded, (size, size), interpolation=interp)


def contrast_stretching_norm(img, n0=128.0, var0=4000.0):
    img_f = img.astype(np.float32)
    out = []
    for c in range(3):
        ch = img_f[..., c]
        m, var = ch.mean(), ch.var() + 1e-6
        coeff = np.sqrt(var0 * (ch - m) ** 2 / var)
        out.append(np.where(ch > m, n0 + coeff, n0 - coeff))
    return np.clip(np.stack(out, axis=-1), 0, 255).astype(np.uint8)


def soft_segment(img, mask_bin, bg_keep=BG_KEEP):
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))
    m = cv2.dilate(mask_bin.astype(np.float32), k)
    m = cv2.GaussianBlur(m, (0, 0), 5)
    weight = bg_keep + (1.0 - bg_keep) * m
    return np.clip(img.astype(np.float32) * weight[..., None], 0, 255).astype(np.uint8)


def run_pipeline(img_path, mask_path):
    bgr = cv2.imread(img_path)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if bgr is None or mask is None:
        return None
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    h, w = rgb.shape[:2]
    if mask.shape != (h, w):
        mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_LINEAR)
    resized = pad_and_resize(rgb, IMG_SIZE, cv2.INTER_AREA, cv2.BORDER_REFLECT)
    mask_r = pad_and_resize(mask, IMG_SIZE, cv2.INTER_LINEAR, cv2.BORDER_CONSTANT)
    mask_bin = (mask_r > 127).astype(np.uint8)
    normed = contrast_stretching_norm(resized)
    return rgb, resized, normed, mask_bin, soft_segment(normed, mask_bin)


print("[INFO] Sampel per sumber:")
for src, grp in df_final.groupby("source"):
    row = grp.iloc[0]
    res = run_pipeline(row["filepath"], row["mask_path"])
    if res is None:
        continue
    rgb, resized, normed, mask_bin, seg = res
    fig, axes = plt.subplots(1, 5, figsize=(18, 4))
    for ax, im, t in zip(axes, [rgb, resized, normed, mask_bin, seg],
                         ["Asli", f"Resize {IMG_SIZE}", "Normalisasi Kontras", "Masker", "Hasil Segmentasi"]):
        ax.imshow(im, cmap="gray" if im.ndim == 2 else None)
        ax.set_title(f"{t}\n{src}" if t == "Asli" else t)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def process_partition(rows):
    import time
    start = time.time()
    out = []
    for idx, image_id, img_path, mask_path in rows:
        out_path = PROC_DIR / f"{image_id}.jpg"
        if not out_path.exists():
            res = run_pipeline(img_path, mask_path)
            if res is None:
                out.append((idx, None))
                continue
            cv2.imwrite(str(out_path), cv2.cvtColor(res[4], cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, 95])
        out.append((idx, str(out_path)))
    yield (len(out), time.time() - start, out)


spark = SparkSession.builder.getOrCreate()
tasks = [(i, r.image_id, r.filepath, r.mask_path) for i, r in df_final.iterrows()]
print(f"[INFO] Spark memproses {len(tasks)} citra: {NUM_WORKERS} worker, {NUM_PARTITIONS} partisi...")
t0 = time.time()
parts = spark.sparkContext.parallelize(tasks, NUM_PARTITIONS).mapPartitions(process_partition).collect()
elapsed_prep = time.time() - t0

print("\n=== BEBAN KERJA PREPROCESSING SPARK ===")
for k, (n, t, _) in enumerate(parts):
    print(f"Partisi {k}: {n} citra, {t:.1f} detik")
print(f"Total waktu preprocessing: {elapsed_prep:.1f} detik")
results = [r for _, _, rows in parts for r in rows]

spark.stop()
import gc
gc.collect()
print("[INFO] Spark ditutup, RAM dibebaskan untuk TensorFlow.")

df_final["filepath"] = [p for _, p in sorted(results)]
failed = df_final["filepath"].isna().sum()
df_final = df_final.dropna(subset=["filepath"]).reset_index(drop=True)
df_final.to_csv(WORK_DIR / f"processed_metadata_{TASK}.csv", index=False)
print(f"[INFO] Selesai. Berhasil: {len(df_final)} | gagal: {failed}")

## cek hasil

In [ ]:
import cv2
import matplotlib.pyplot as plt
samples = df_final.head(5)

print("=== Pengecekan Dimensi 5 Gambar Teratas ===")
plt.figure(figsize=(15, 4))
for i, (index, row) in enumerate(samples.iterrows()):
    img_path = row['filepath']
    # Membaca gambar dari direktori offline preprocessing
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    # Mengekstrak dimensi gambar
    tinggi, lebar, _ = img.shape
    nama_file = img_path.split('/')[-1]
    # Mencetak info dimensi ke console
    print(f"[{nama_file}] -> Lebar: {lebar}, Tinggi: {tinggi}")
    # Menampilkan gambar pada plot
    plt.subplot(1, 5, i + 1)
    plt.title(f"Sampel {i+1}\n{lebar}x{tinggi}")
    plt.imshow(img)
    plt.axis('off')

plt.tight_layout()
plt.show()

# split dataset (pakai split resmi) dan pipeline data di RAM

In [ ]:
import math
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.applications.resnet50 import preprocess_input
from sklearn.utils import resample

train_df = df_final[df_final["split"] == "train"].reset_index(drop=True)
val_df = df_final[df_final["split"] == "val"].reset_index(drop=True)
test_df = df_final[df_final["split"] == "test"].reset_index(drop=True)

assert not (set(train_df.image_id) & set(val_df.image_id)), "Bocor train-val"
assert not (set(train_df.image_id) & set(test_df.image_id)), "Bocor train-test"
assert not (set(val_df.image_id) & set(test_df.image_id)), "Bocor val-test"

print("=== RINGKASAN PEMBAGIAN DATA ===")
print(f"Train: {len(train_df)} | Validation: {len(val_df)} | Test: {len(test_df)}\n")

CLASS_NAMES = sorted(df_final["label"].unique())
class_idx = {c: i for i, c in enumerate(CLASS_NAMES)}
assert len(CLASS_NAMES) == NUM_CLASSES

train_df["pos"] = np.arange(len(train_df))
max_count = train_df["label"].value_counts().max()
dfs_balanced = []
for kelas in CLASS_NAMES:
    df_kelas = train_df[train_df["label"] == kelas]
    if len(df_kelas) < max_count:
        df_kelas = resample(df_kelas, replace=True, n_samples=max_count, random_state=42)
    dfs_balanced.append(df_kelas)
train_df_balanced = pd.concat(dfs_balanced).sample(frac=1, random_state=42).reset_index(drop=True)
print("=== DISTRIBUSI TRAIN SET (OVERSAMPLED) ===")
print(train_df_balanced["label"].value_counts().to_string())


workers = min(os.cpu_count() or 4, 16)


def load_rgb(path):
    return cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)


def load_array(df):
    with ThreadPoolExecutor(max_workers=workers) as ex:
        return np.stack(list(tqdm(ex.map(load_rgb, df["filepath"]), total=len(df))))


print("\n[INFO] Memuat citra ke RAM (uint8)...")
X_train_uni = load_array(train_df)
X_train = X_train_uni[train_df_balanced["pos"].to_numpy()]
del X_train_uni
X_val, X_test = load_array(val_df), load_array(test_df)
assert X_train.shape[1:] == (IMG_SIZE, IMG_SIZE, 3)

y_train_idx = train_df_balanced["label"].map(class_idx).to_numpy()
y_val_idx = val_df["label"].map(class_idx).to_numpy()
y_test_idx = test_df["label"].map(class_idx).to_numpy()
eye = np.eye(NUM_CLASSES, dtype=np.float32)
print(f"X_train {X_train.shape} | X_val {X_val.shape} | X_test {X_test.shape}")

augment = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(15 / 360, fill_mode="reflect"),
    layers.RandomZoom(0.15, fill_mode="reflect"),
    layers.RandomTranslation(0.1, 0.1, fill_mode="reflect"),
    layers.RandomBrightness(0.3, value_range=(0, 255)),
])


class BatchSeq(tf.keras.utils.Sequence):
    def __init__(self, X, y_idx, batch_size, train=False):
        super().__init__()
        self.X, self.y, self.bs, self.train = X, eye[y_idx], batch_size, train
        self.order = np.arange(len(X))
        if train:
            np.random.shuffle(self.order)

    def __len__(self):
        return math.ceil(len(self.X) / self.bs)

    def __getitem__(self, i):
        if i >= len(self):
            raise IndexError(i)
        idx = self.order[i * self.bs:(i + 1) * self.bs]
        x = tf.constant(self.X[idx], dtype=tf.float32)
        if self.train:
            x = augment(x, training=True)
        return preprocess_input(x).numpy(), self.y[idx]

    def on_epoch_end(self):
        if self.train:
            np.random.shuffle(self.order)


train_data = BatchSeq(X_train, y_train_idx, BATCH_SIZE, train=True)
val_data = BatchSeq(X_val, y_val_idx, BATCH_SIZE)
test_data = BatchSeq(X_test, y_test_idx, BATCH_SIZE)
print(f"Batch: train {len(train_data)} | val {len(val_data)} | test {len(test_data)}")

## cek hasil

In [ ]:
# ==========================================
# CEK HASIL GENERATOR TAHAP 4 & SPESIFIKASI
# ==========================================

import matplotlib.pyplot as plt
import numpy as np

# 1. Mengambil 1 batch data dari generator
images, labels = train_data[0]

# 2. Mengambil pemetaan nama kelas
class_labels = CLASS_NAMES

# 3. Menampilkan Spesifikasi Teknis
print("=== SPESIFIKASI BATCH GAMBAR TAHAP 4 ===")
print(f"Bentuk Array Batch   : {images.shape} -> (Batch Size, Tinggi, Lebar, Channel RGB)")
print(f"Resolusi per Gambar  : {images.shape[1]} x {images.shape[2]} piksel")
print(f"Bentuk Label Batch   : {labels.shape} -> (Batch Size, Jumlah Kelas)")
print(f"Tipe Data Array      : {images.dtype}")
print(f"Rentang Nilai Piksel : {np.min(images):.2f} sampai {np.max(images):.2f} (Bukti preprocess_input ResNet50 aktif)")
print(f"Nama Kelas           : {class_labels}")
print("========================================\n")


# 4. Visualisasi 5 Gambar Pertama
fig, axes = plt.subplots(1, 5, figsize=(20, 5))


for i in range(5):

    ax = axes[i]

    # ===============================
    # BALIKKAN PREPROCESS RESNET50
    # AGAR GAMBAR NORMAL SAAT DITAMPILKAN
    # ===============================

    img_show = images[i].copy()

    # Tambahkan kembali mean ImageNet
    img_show = img_show + [103.939, 116.779, 123.68]

    # BGR -> RGB
    img_show = img_show[..., ::-1]

    # Batasi nilai
    img_show = np.clip(img_show, 0, 255)

    # ubah ke range matplotlib
    img_show = img_show / 255.0


    ax.imshow(img_show)


    # Label: argmax dari one-hot vector (karena class_mode='categorical')
    label_idx = int(np.argmax(labels[i]))
    nama_kelas = class_labels[label_idx]


    ax.set_title(
        f"Gambar {i+1}\nKelas: {nama_kelas}\n(Label: {label_idx})",
        fontsize=12
    )

    ax.axis('off')


plt.tight_layout()
plt.show()

# Modeling

In [ ]:
# ==========================================
# TAHAP 5: KONSTRUKSI ARSITEKTUR MODEL (MULTI-GPU) + TRIPLET ATTENTION
# ==========================================

import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import (
    Dense, GlobalAveragePooling2D, GlobalMaxPooling2D,
    BatchNormalization, Dropout, Concatenate, Input,
    Conv2D, Activation, Lambda, Multiply
)
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2
import tensorflow.keras.backend as K


# ---- TRIPLET ATTENTION MODULE ----

class ZPool(tf.keras.layers.Layer):
    """Z-Pool: Concat max dan average sepanjang axis tertentu."""
    def call(self, x):
        max_out = tf.reduce_max(x, axis=-1, keepdims=True)
        avg_out = tf.reduce_mean(x, axis=-1, keepdims=True)
        return tf.concat([max_out, avg_out], axis=-1)


class AttentionGate(tf.keras.layers.Layer):
    """Satu branch attention: ZPool -> Conv2D 3x3 -> Sigmoid."""
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.zpool = ZPool()
        self.conv = Conv2D(1, kernel_size=3, padding='same', use_bias=False)
        self.sigmoid = Activation('sigmoid')

    def call(self, x):
        out = self.zpool(x)
        out = self.conv(out)
        out = self.sigmoid(out)
        return out


class TripletAttention(tf.keras.layers.Layer):
    """
    Triplet Attention Module (WACV 2021)
    3 branch yang menangkap interaksi:
      - Branch 1: Channel x Height  (rotasi C<->H)
      - Branch 2: Channel x Width   (rotasi C<->W)
      - Branch 3: Height x Width    (spatial attention standar)
    """
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.gate_ch = AttentionGate()   # Branch 1: C-H interaction
        self.gate_cw = AttentionGate()   # Branch 2: C-W interaction
        self.gate_hw = AttentionGate()   # Branch 3: H-W interaction (spatial)

    def call(self, x):
        # x shape: (B, H, W, C)

        # --- Branch 1: C x H ---
        # Permute: (B, H, W, C) -> (B, C, W, H)
        x_perm1 = tf.transpose(x, perm=[0, 3, 2, 1])
        attn1 = self.gate_ch(x_perm1)
        x_perm1 = x_perm1 * attn1
        # Kembalikan: (B, C, W, H) -> (B, H, W, C)
        x_perm1 = tf.transpose(x_perm1, perm=[0, 3, 2, 1])

        # --- Branch 2: C x W ---
        # Permute: (B, H, W, C) -> (B, H, C, W)
        x_perm2 = tf.transpose(x, perm=[0, 1, 3, 2])
        attn2 = self.gate_cw(x_perm2)
        x_perm2 = x_perm2 * attn2
        # Kembalikan: (B, H, C, W) -> (B, H, W, C)
        x_perm2 = tf.transpose(x_perm2, perm=[0, 1, 3, 2])

        # --- Branch 3: H x W (Spatial Attention standar) ---
        attn3 = self.gate_hw(x)
        x_branch3 = x * attn3

        # Gabungkan 3 branch (rata-rata)
        out = (x_perm1 + x_perm2 + x_branch3) / 3.0
        return out


# ---- BANGUN MODEL ----

# 1. MENGAKTIFKAN DUAL GPU (TESLA T4 x2)
strategy = tf.distribute.MirroredStrategy()
print(f"[INFO] Jumlah GPU Tersinkronisasi: {strategy.num_replicas_in_sync}")

# 2. Membangun model DI DALAM scope Multi-GPU
with strategy.scope():
    base_model = ResNet50(
        weights='imagenet',
        include_top=False,
        input_shape=(224, 224, 3)
    )
    base_model.trainable = False

    # =====================================================
    # ARSITEKTUR BARU: TripletAttention di conv4 (14x14)
    # Resolusi 14x14 jauh lebih efektif untuk attention
    # dibanding 7x7 di output akhir ResNet50
    # =====================================================
    
    # Ambil output dari conv4_block6 (14x14x1024)
    conv4_out = base_model.get_layer('conv4_block6_out').output
    
    # Terapkan Triplet Attention pada resolusi 14x14
    attended = TripletAttention(name='triplet_attention')(conv4_out)
    
    # Lanjutkan ke conv5 blocks menggunakan layer ResNet asli
    # Kita buat sub-model conv5 terpisah
    conv5_input = base_model.get_layer('conv5_block1_1_conv').input
    conv5_output = base_model.output  # (B, 7, 7, 2048)
    
    # Bangun model: Input -> conv1-conv4 -> TripletAttention -> conv5 -> Output
    # Karena conv5 sudah terhubung di graph ResNet, kita gunakan approach berbeda:
    # Ambil output akhir ResNet (7x7x2048) lalu tambahkan attention skip connection
    
    x_main = base_model.output  # (B, 7, 7, 2048) - fitur utama dari seluruh ResNet
    
    # Attention-weighted conv4 features (14x14x1024)
    # Pooling attended features ke ukuran yang sama
    attended_avg = GlobalAveragePooling2D(name='att_avg_pool')(attended)  # (B, 1024)
    
    # Main ResNet features (7x7x2048)
    avg_pool = GlobalAveragePooling2D()(x_main)   # (B, 2048)
    max_pool = GlobalMaxPooling2D()(x_main)        # (B, 2048)
    
    # Gabungkan: fitur ResNet + fitur attention-enhanced
    x = Concatenate()([avg_pool, max_pool, attended_avg])  # (B, 2048+2048+1024 = 5120)
    
    x = BatchNormalization()(x)

    x = Dense(512, activation='relu', kernel_regularizer=l2(0.001))(x)
    x = Dropout(0.5)(x)
    x = Dense(256, activation='relu', kernel_regularizer=l2(0.001))(x)
    x = Dropout(0.4)(x)

    # OUTPUT: softmax float32 sesuai NUM_CLASSES
    predictions = Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)
    model = Model(inputs=base_model.input, outputs=predictions)

print("Ringkasan Arsitektur Model (Multi-GPU + Triplet Attention @ Conv4 14x14 - 3 Kelas):")
model.summary()

# Training

In [ ]:
# ==========================================================
# TAHAP 6: PELATIHAN 2 FASE (MULTI-GPU, FOCAL, MICRO-UNFREEZE)
# ==========================================================

import time
import numpy as np
import tensorflow as tf
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)

# ----------------------------------------------------
# 1. METRIK, FOCAL LOSS & CALLBACKS
# ----------------------------------------------------
# CategoricalFocalCrossentropy untuk klasifikasi 3-kelas
# Alpha 0.25 (default) cocok karena data sudah di-oversample seimbang
focal_loss = tf.keras.losses.CategoricalFocalCrossentropy(
    gamma=2.0, 
    alpha=0.25
)

metrics_list = [
    'accuracy',
    tf.keras.metrics.AUC(name='auc', multi_label=True, num_labels=NUM_CLASSES),
    tf.keras.metrics.AUC(curve='PR', name='pr_auc', multi_label=True, num_labels=NUM_CLASSES)
]

early_stop = EarlyStopping(
    monitor='val_pr_auc', 
    mode='max',
    patience=10, 
    restore_best_weights=True,
    verbose=1
)

lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss', 
    mode='min',
    factor=0.5, 
    patience=3, 
    min_lr=1e-7,
    verbose=1
)

checkpoint = ModelCheckpoint(
    filepath=CKPT_PATH,
    monitor='val_pr_auc',
    mode='max',
    save_best_only=True,
    verbose=1
)

callbacks_list = [lr_scheduler, early_stop, checkpoint]

# ----------------------------------------------------
# 2. FASE 1: WARM-UP (Melatih Custom Head saja)
# ----------------------------------------------------
base_model.trainable = False

with strategy.scope():
    model.compile(
        optimizer=AdamW(learning_rate=1e-4, weight_decay=1e-4),
        loss=focal_loss,
        metrics=metrics_list
    )

print("[INFO] Memulai Fase 1: Warm-up Custom Head...")
start_time_f1 = time.time()

history_f1 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10, 
    callbacks=callbacks_list
)

waktu_f1 = (time.time() - start_time_f1) / 60
print(f"Waktu Pelatihan Fase 1: {waktu_f1:.2f} menit\n")

# ----------------------------------------------------
# 3. FASE 2: MICRO-UNFREEZING (HANYA conv5_block3)
# ----------------------------------------------------
print("[INFO] Membuka conv5_block3, Kunci BatchNorm...")

with strategy.scope():
    base_model.trainable = True

    for layer in base_model.layers:
        # Kunci MATI seluruh layer BatchNormalization
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False
            
        # Buka HANYA blok terdalam
        elif "conv5_block3" in layer.name:
            layer.trainable = True
            
        # Kunci sisanya
        else:
            layer.trainable = False

    # Learning Rate diturunkan perlahan
    model.compile(
        optimizer=AdamW(learning_rate=5e-6, weight_decay=1e-4),
        loss=focal_loss,
        metrics=metrics_list
    )

print("[INFO] Memulai Fase 2: Micro Fine-Tuning...")
start_time_f2 = time.time()

history_f2 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=30, 
    callbacks=callbacks_list
)

waktu_f2 = (time.time() - start_time_f2) / 60
print(f"Waktu Pelatihan Fase 2: {waktu_f2:.2f} menit\n")

# ----------------------------------------------------
# 4. EVALUASI BLIND TEST PADA DATA UJI (TEST SET)
# ----------------------------------------------------
print(f"[HASIL] Total Waktu Pelatihan: {waktu_f1+waktu_f2:.2f} menit")

# Menggunakan test_data yang sama sekali tidak disentuh saat training
print("\n[INFO] Menjalankan Evaluasi Blind Test...")
eval_results = model.evaluate(test_data)

final_loss = eval_results[0]
final_accuracy = eval_results[1]
final_auc = eval_results[2]
final_pr_auc = eval_results[3]

print(f"\n--- HASIL BLIND TEST AKHIR ---")
print(f"Akurasi Test : {final_accuracy * 100:.2f}%")
print(f"Loss Test    : {final_loss:.4f}")
print(f"AUC          : {final_auc:.4f}")
print(f"PR-AUC       : {final_pr_auc:.4f}")
print("------------------------------\n")

# Evaluasi

In [ ]:
# ==========================================
# TAHAP 7: EVALUASI MODEL & VISUALISASI (3-KELAS)
# ==========================================

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix
)

# ----------------------------------------------------
# 1. MENGGABUNGKAN RIWAYAT PELATIHAN (FASE 1 & FASE 2)
# ----------------------------------------------------
acc = history_f1.history['accuracy'] + history_f2.history['accuracy']
val_acc = (
    history_f1.history['val_accuracy'] + 
    history_f2.history['val_accuracy']
)
loss = history_f1.history['loss'] + history_f2.history['loss']
val_loss = (
    history_f1.history['val_loss'] + 
    history_f2.history['val_loss']
)

batas_fase = len(history_f1.history['accuracy']) - 1

# ----------------------------------------------------
# 2. VISUALISASI GRAFIK AKURASI DAN LOSS
# ----------------------------------------------------
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.axvline(
    x=batas_fase, color='red', linestyle='--', 
    label='Mulai Fine-Tuning'
)
plt.title('Pergerakan Akurasi Model')
plt.xlabel('Epoch')
plt.ylabel('Akurasi')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.axvline(
    x=batas_fase, color='red', linestyle='--', 
    label='Mulai Fine-Tuning'
)
plt.title('Pergerakan Loss Model')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

# ----------------------------------------------------
# 3. EVALUASI VALIDATION SET (3-KELAS)
# ----------------------------------------------------
print("\n[INFO] Memuat model terbaik dari checkpoint fisik...")
# ============================================================
# FIX: Daftarkan semua custom layer SEBELUM load_model
# Keras perlu tahu class TripletAttention, ZPool, AttentionGate
# agar bisa membangun ulang model dari file .keras
# ============================================================
best_model = tf.keras.models.load_model(
    CKPT_PATH,
    custom_objects={
        'TripletAttention': TripletAttention,
        'AttentionGate': AttentionGate,
        'ZPool': ZPool,
    }
)

print("[INFO] Mengevaluasi Validation Set...")

# Prediksi menghasilkan array (N, 3) — probabilitas setiap kelas
Y_pred_val = best_model.predict(
    val_data, verbose=0
)

# Konversi probabilitas ke label kelas (argmax)
y_pred_classes_val = np.argmax(Y_pred_val, axis=1)
y_true_val = y_val_idx
target_names = CLASS_NAMES

# Pastikan panjang prediksi = panjang label asli
y_pred_classes_val = y_pred_classes_val[:len(y_true_val)]

print("=== CLASSIFICATION REPORT (VALIDATION SET) ===")
print(
    classification_report(
        y_true_val, y_pred_classes_val, target_names=target_names
    )
)

# ----------------------------------------------------
# 4. BLIND TEST & EVALUASI TEST SET
# ----------------------------------------------------
print("\n[INFO] Mengevaluasi Test Set (Blind Test)...")

Y_pred_test = best_model.predict(
    test_data, verbose=0
)

y_pred_classes_test = np.argmax(Y_pred_test, axis=1)
y_true_test = y_test_idx

# Pastikan panjang prediksi = panjang label asli
y_pred_classes_test = y_pred_classes_test[:len(y_true_test)]

print("=== CLASSIFICATION REPORT (TEST SET) ===")
print(
    classification_report(
        y_true_test, y_pred_classes_test, target_names=target_names
    )
)

# ----------------------------------------------------
# 5. VISUALISASI CONFUSION MATRIX BERDAMPINGAN (3x3)
# ----------------------------------------------------
cm_val = confusion_matrix(y_true_val, y_pred_classes_val)
cm_test = confusion_matrix(y_true_test, y_pred_classes_test)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot Validation CM
sns.heatmap(
    cm_val, annot=True, fmt='d', cmap='Blues', 
    xticklabels=target_names, yticklabels=target_names,
    annot_kws={"size": 12}, ax=axes[0]
)
axes[0].set_title('Confusion Matrix - Validation Set', fontsize=14)
axes[0].set_ylabel('Label Aktual', fontsize=12)
axes[0].set_xlabel('Label Prediksi', fontsize=12)

# Plot Test CM
sns.heatmap(
    cm_test, annot=True, fmt='d', cmap='Greens', 
    xticklabels=target_names, yticklabels=target_names,
    annot_kws={"size": 12}, ax=axes[1]
)
axes[1].set_title('Confusion Matrix - Test Set', fontsize=14)
axes[1].set_ylabel('Label Aktual', fontsize=12)
axes[1].set_xlabel('Label Prediksi', fontsize=12)

plt.tight_layout()
plt.show()